In [1]:
# Jupyter Notebook for baseline model training
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from src.data_loader import load_and_validate
from src.features import add_time_of_day_feature, train_test_split_stratified

# Reusing Day 1's validated loader — if the data is malformed, we find
# out here, at cell 1, not mid-training.
df = load_and_validate()
df = add_time_of_day_feature(df)
df.shape

(284807, 32)

In [2]:
# Splitting data in train and test sets, stratified by the target variable
X_train, X_test, y_train, y_test = train_test_split_stratified(df)

print(f"Train: {len(X_train)} rows, {y_train.sum()} fraud ({y_train.mean():.4%})")
print(f"Test:  {len(X_test)} rows, {y_test.sum()} fraud ({y_test.mean():.4%})")

Train: 227845 rows, 394 fraud (0.1729%)
Test:  56962 rows, 98 fraud (0.1720%)


## Model Comparison Setup

This notebook trains and compares three approaches to fraud detection:

1. **Logistic Regression** — simple, fast, interpretable supervised baseline
2. **XGBoost** — stronger supervised model, likely the best performer on offline metrics
3. **Isolation Forest** — unsupervised anomaly detection, trained only on non-fraud transactions

All three are evaluated on precision, recall, and PR-AUC — not accuracy, which is meaningless at a 0.17% fraud rate (see `01_eda.ipynb`).

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from src.evaluate import evaluate_predictions, print_confusion_summary

# Pipeline bundles scaling + model into one object. StandardScaler
# rescales every feature to have mean 0 and standard deviation 1 —
# putting Amount, Time, and the V columns on comparable scales, which
# is what lbfgs needs to converge properly.
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    )),
])

lr_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](31,)","['Time','V1','V2',...,'V28','Amount','hour_of_day']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [4]:
lr_pred = lr_model.predict(X_test)
lr_scores = lr_model.predict_proba(X_test)[:, 1]

lr_results = evaluate_predictions(y_test, lr_pred, lr_scores, model_name="Logistic Regression")
print(lr_results)

print_confusion_summary(y_test, lr_pred, model_name="Logistic Regression")

{'model': 'Logistic Regression', 'precision': 0.0559748427672956, 'recall': 0.9081632653061225, 'f1': 0.10545023696682465, 'pr_auc': 0.7236827200635098}
--- Logistic Regression ---
True Positives  (fraud correctly caught):        89
False Negatives (fraud missed — real loss):      9
False Positives (legit txns wrongly flagged):    1501
True Negatives  (legit txns correctly cleared):  55363


In [5]:
# Training XGBoost model
from xgboost import XGBClassifier
from src.evaluate import evaluate_predictions, print_confusion_summary

# Computed from the actual training data, not hardcoded — this is the
# standard recommended starting point for scale_pos_weight.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",   # optimize/track PR-AUC directly during training,
                            # consistent with how we're evaluating everything else
    random_state=42,
    n_estimators=100,      # number of boosting rounds — a reasonable default
                            # to start from, not yet tuned
)

xgb_model.fit(X_train, y_train)

scale_pos_weight: 577.3


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [6]:
xgb_pred = xgb_model.predict(X_test)
xgb_scores = xgb_model.predict_proba(X_test)[:, 1]

xgb_results = evaluate_predictions(y_test, xgb_pred, xgb_scores, model_name="XGBoost")
print(xgb_results)

print_confusion_summary(y_test, xgb_pred, model_name="XGBoost")

{'model': 'XGBoost', 'precision': 0.8631578947368421, 'recall': 0.8367346938775511, 'f1': 0.8497409326424871, 'pr_auc': 0.8808140194846128}
--- XGBoost ---
True Positives  (fraud correctly caught):        82
False Negatives (fraud missed — real loss):      16
False Positives (legit txns wrongly flagged):    13
True Negatives  (legit txns correctly cleared):  56851


In [7]:
# Training Isolation Forest model
from sklearn.ensemble import IsolationForest
from src.evaluate import evaluate_predictions, print_confusion_summary
# Training only on non-fraud examples — the model never sees a single
# labeled fraud case, and instead learns what "normal" looks like.
X_train_normal = X_train[y_train == 0]
print(f"Training Isolation Forest on {len(X_train_normal)} non-fraud transactions only")

iso_model = IsolationForest(
    contamination=0.0017,  # expected proportion of anomalies — see note below
    n_estimators=100,
    random_state=42,
)
iso_model.fit(X_train_normal)

Training Isolation Forest on 227451 non-fraud transactions only


,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.0017
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [8]:
iso_raw_pred = iso_model.predict(X_test)         # returns 1 (normal) or -1 (anomaly)
iso_pred = (iso_raw_pred == -1).astype(int)       # convert to our 0/1 fraud convention

# decision_function: higher = more "normal", lower/negative = more anomalous.
# We negate it so higher score = more fraud-like, matching the convention
# used by lr_scores and xgb_scores (higher = more likely fraud), which is
# what evaluate_predictions()'s PR-AUC calculation expects.
iso_scores = -iso_model.decision_function(X_test)

iso_results = evaluate_predictions(y_test, iso_pred, iso_scores, model_name="Isolation Forest")
print(iso_results)
print_confusion_summary(y_test, iso_pred, model_name="Isolation Forest")

{'model': 'Isolation Forest', 'precision': 0.1889763779527559, 'recall': 0.24489795918367346, 'f1': 0.21333333333333335, 'pr_auc': 0.10346954273308215}
--- Isolation Forest ---
True Positives  (fraud correctly caught):        24
False Negatives (fraud missed — real loss):      74
False Positives (legit txns wrongly flagged):    103
True Negatives  (legit txns correctly cleared):  56761


In [9]:
from src.evaluate import compare_models

comparison_table = compare_models([lr_results, xgb_results, iso_results])
comparison_table

,precision,recall,f1,pr_auc
model,,,,
Logistic Regression,0.055975,0.908163,0.105450,0.723683
XGBoost,0.863158,0.836735,0.849741,0.880814
Isolation Forest,0.188976,0.244898,0.213333,0.103470


## Model Comparison — Interpretation

| Model | Precision | Recall | F1 | PR-AUC |
|---|---|---|---|---|
| Logistic Regression | 5.6% | 90.8% | 0.105 | 0.724 |
| XGBoost | 86.3% | 83.7% | 0.850 | 0.881 |
| Isolation Forest | 18.9% | 24.5% | 0.213 | 0.103 |

**XGBoost is the clear leader on offline metrics.** It dominates Isolation Forest outright (better on every metric) and offers a far more usable precision/recall balance than Logistic Regression — 86.3% precision vs 5.6% means the difference between 13 false alarms and 1,501 false alarms on this test set alone, at a relatively small recall cost (83.7% vs 90.8%).

**Logistic Regression's extreme recall-skew is a direct consequence of `class_weight='balanced'`**, not an inherent property of linear models — it illustrates how aggressively that setting trades precision away. It remains useful as a baseline and as a reference point for "maximum recall, if a business were willing to tolerate very high false-positive volume."

**Isolation Forest underperformed substantially in this baseline configuration** (PR-AUC 0.103, barely above the dataset's fraud rate). This is a genuine, honest result from an *untuned* baseline trained with a tight `contamination=0.0017` setting — it should not be read as "unsupervised anomaly detection is fundamentally unsuitable here." Untuned hyperparameters and PCA features that may not suit random-split isolation well are both plausible contributors. A fairer test would tune `contamination`/`n_estimators`, or restrict to the strongest EDA-correlated features, before drawing firm conclusions.

**Why not just declare XGBoost the winner and stop:** offline metrics don't capture everything relevant to a production decision. XGBoost, as a supervised model, is fundamentally limited to patterns present in its labeled training data — it cannot flag a genuinely novel fraud pattern it's never seen an example of. Isolation Forest's real value proposition isn't matching XGBoost's accuracy; it's catching fraud *XGBoost's labels never taught it to expect*. This baseline doesn't demonstrate that value yet, but the architectural argument for including both approaches in a production system — supervised for known patterns, unsupervised as a complementary safety net for unknown ones — still holds independent of this specific result.

In [10]:
# Verifying script
from src.train import load_model

model = load_model("xgboost")
print(type(model))
print(model.predict_proba)

<class 'xgboost.sklearn.XGBClassifier'>
<bound method XGBClassifier.predict_proba of XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)>
